In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

DATA_DIR = Path("../data/news-portal-user-interactions-by-globocom")
CLICKS_DIR = DATA_DIR / "clicks" / "clicks"

articles_metadata_path = DATA_DIR / "articles_metadata.csv"
clicks_sample_path = DATA_DIR / "clicks_sample.csv"
embeddings_path = DATA_DIR / "articles_embeddings.pickle"

# Chargement
articles_metadata = pd.read_csv(articles_metadata_path)
clicks_sample = pd.read_csv(clicks_sample_path)

with open(embeddings_path, "rb") as f:
    embeddings = pickle.load(f)

print("articles_metadata shape:", articles_metadata.shape)
print("clicks_sample shape:", clicks_sample.shape)
print("type embeddings:", type(embeddings))

/tmp/ipykernel_61715/1003454302.py:22: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  embeddings = pickle.load(f)


articles_metadata shape: (364047, 5)
clicks_sample shape: (1883, 12)
type embeddings: <class 'numpy.ndarray'>


In [7]:
def get_user_history(user_id: int, clicks_df: pd.DataFrame) -> np.ndarray:
    """
    Retourne les article_id déjà consultés par l'utilisateur.
    """
    return clicks_df.loc[
        clicks_df["user_id"] == user_id,
        "click_article_id"
    ].unique()

In [8]:
def build_user_profile(user_history: np.ndarray, embeddings: np.ndarray) -> np.ndarray:
    """
    Moyenne des embeddings des articles vus.
    """
    user_vectors = embeddings[user_history]
    return user_vectors.mean(axis=0).reshape(1, -1)

In [9]:
def recommend_articles(
    user_id: int,
    clicks_df: pd.DataFrame,
    embeddings: np.ndarray,
    top_k: int = 5
):
    history = get_user_history(user_id, clicks_df)

    if len(history) == 0:
        raise ValueError(f"Aucun historique pour user {user_id}")

    user_profile = build_user_profile(history, embeddings)

    similarities = cosine_similarity(user_profile, embeddings)[0]

    candidate_ids = np.argsort(similarities)[::-1]

    recommendations = [
        article_id
        for article_id in candidate_ids
        if article_id not in history
    ][:top_k]

    return recommendations

In [10]:
test_user = 681

history = get_user_history(test_user, clicks_sample)
reco = recommend_articles(test_user, clicks_sample, embeddings)

print("Historique :", history)
print("Top 5 reco :", reco)

Historique : [ 68866 284847  30970 258386 283177 114349 363291 361700 299837 272947
 270850 273355 271672 141945 308807 260988 332114 136590 141517 270225
 342239 288431 198420 288984]
Top 5 reco : [np.int64(217852), np.int64(363057), np.int64(272247), np.int64(261014), np.int64(285757)]


In [11]:
print("Articles déjà vus :", set(history))
print("Intersection :", set(history).intersection(set(reco)))

Articles déjà vus : {np.int64(68866), np.int64(270850), np.int64(136590), np.int64(270225), np.int64(198420), np.int64(363291), np.int64(283177), np.int64(114349), np.int64(284847), np.int64(288431), np.int64(272947), np.int64(271672), np.int64(299837), np.int64(308807), np.int64(273355), np.int64(141517), np.int64(258386), np.int64(332114), np.int64(288984), np.int64(342239), np.int64(361700), np.int64(141945), np.int64(30970), np.int64(260988)}
Intersection : set()


In [12]:
articles_metadata.loc[
    articles_metadata["article_id"].isin(reco)
]

,article_id,category_id,created_at_ts,publisher_id,words_count
217852,217852,352,1506793697000,0,191
261014,261014,395,1512997291000,0,150
272247,272247,399,1504091532000,0,198
285757,285757,412,1502565301000,0,183
363057,363057,455,1514723071000,0,162
